# Comparing Regressors

*Section 5 — Regression, notebook 6 of 6.* Previous: `05_multivariate_regression`. Next: `06_change_detection/01_create_baseline_class`.

**You will learn:**

- how to compare six scikit-learn regressors with a repeated k-fold cross-validation (`rsgispy.regression.regsklearn.perform_kfold_fit`);
- how to add data scaling to a regressor with a scikit-learn `Pipeline`;
- how to summarise the k-fold accuracy metrics with `summarise_kfold_metrics` and pandas to choose the 'best' regressor;
- how to compare the residuals of two regressors with `plot_residuals`.

**Before you start:** notebook 5, which introduces the k-fold assessment; nothing from it needs to have been run.

**Data:** `../data/lidar/Forest_Plot_Metrics_LassoLars_Sel.csv`.

**Run time:** about 1 minute.

## Overview

It is commonly the case that is it no possible to predict which regressor will produce the best results. Therefore, we need to try each of the regressors and compare the results. For this analysis we will not optimise the parameters for the regressors which should be done but this was demonstrated in the previous notebook and significantly increases the compute time for the notebook. 

# 1. Imports

In [1]:
import os

import pandas
import rsgispy.regression.regsklearn as rsk
from rsgispy.regression.regmetrics import plot_residuals
from rsgispy.tools.strings import check_str
from sklearn.cross_decomposition import PLSRegression
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.kernel_ridge import KernelRidge
from sklearn.linear_model import ElasticNet, LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# 2. Read the input plot data 

In [2]:
# Open the CSV file as a Pandas data frame - the df variable.
df = pandas.read_csv("../data/lidar/Forest_Plot_Metrics_LassoLars_Sel.csv", index_col=0)

# Get a list of the columns within the df dataframe
cols = list(df.columns)

# Get the indepedent predictor column names
ind_vars = cols[6:]

# Get the dependent response column names
dep_vars = cols[3:6]

# Get the predictor variables and dependent variables
# from the dataframe as numpy arrays
x = df[ind_vars].values
y = df[dep_vars].values

# 3. Create output directory

In [3]:
# All the files created by this notebook are written to the outputs directory.
out_dir = os.path.join("outputs", "compare_multivar_reg_outputs")
os.makedirs(out_dir, exist_ok=True)

# 4. K-fold assessment

As in notebook 5, each regressor is assessed with `rsgispy.regression.regsklearn.perform_kfold_fit`: a repeated k-fold cross-validation (5 folds repeated 20 times, 100 model fits) in which each fold is predicted by a copy of the regressor fitted to the other folds, and the accuracy of each fold is calculated with `rsgispy.regression.regmetrics.calc_reg_metrics`. It returns, for each dependent variable, a DataFrame with the metrics of each fold (r2, rmse, mae, bias, explained_variance, pearson_r, mse, and the normalised RMSE and bias as a percentage of the mean, nrmse and nbias, and of the range, nrmse_range) and a DataFrame of the observed and predicted values (the residuals) of all the folds. With `out_dir` these are also written to CSV files, named with `out_prefix` (here `kfold_<regressor>`) and the dependent variable. The same `rnd_seed` is used for every regressor, so they are all tested on the same folds.

> **RSGISLib users:** this replaces `rsgislib.regression.regresssklearn.perform_kfold_fit`. Instead of a `data_scaler` argument, put a scaler and the regressor in a scikit-learn `Pipeline`, so the scaler is fitted to the training folds only. RSGISLib also reported the median absolute error and a bias-variance decomposition, which are not calculated.

In [4]:
def run_kfold(regrs_obj, alg_name):
    """Run the k-fold assessment of a regressor, writing its outputs to out_dir."""
    return rsk.perform_kfold_fit(
        regrs_obj,
        x,
        y,
        dep_vars,
        n_splits=5,
        n_repeats=20,
        rnd_seed=0,
        out_dir=out_dir,
        out_prefix="kfold_{}".format(alg_name),
    )

## 4.1 Scaling the data

The k-nearest neighbour regressor uses the distances between samples, so the predictor variables need to be on the same scale. A scikit-learn `Pipeline` (made with `make_pipeline`) runs a `StandardScaler` before the regressor: when the pipeline is fitted to the training folds, the scaler is fitted to those samples only and then applied to the held-out fold.

> **RSGISLib users:** this replaces the `data_scaler` argument of `perform_kfold_fit`, where the scaler was fitted to all the samples before the k-fold.

# 5. KFold Extra Trees

In [5]:
skregrs_obj = ExtraTreesRegressor(random_state=0)
et_metrics, et_residuals = run_kfold(skregrs_obj, "ET")

  0%|          | 0/100 [00:00<?, ?it/s]

# 6. KFold Kernel Ridge

In [6]:
skregrs_obj = KernelRidge()
kr_metrics, kr_residuals = run_kfold(skregrs_obj, "KR")

  0%|          | 0/100 [00:00<?, ?it/s]

# 7. KFold ElasticNet

In [7]:
skregrs_obj = ElasticNet()
en_metrics, en_residuals = run_kfold(skregrs_obj, "EN")

  0%|          | 0/100 [00:00<?, ?it/s]

# 8. KFold K-Nearest Neighbour

In [8]:
skregrs_obj = make_pipeline(StandardScaler(), KNeighborsRegressor())
knn_metrics, knn_residuals = run_kfold(skregrs_obj, "KNN")

  0%|          | 0/100 [00:00<?, ?it/s]

# 9. KFold PLSRegression

In [9]:
skregrs_obj = PLSRegression()
pls_metrics, pls_residuals = run_kfold(skregrs_obj, "PLS")

  0%|          | 0/100 [00:00<?, ?it/s]

# 10. KFold Linear Regression

In [10]:
skregrs_obj = LinearRegression()
ols_metrics, ols_residuals = run_kfold(skregrs_obj, "OLS")

  0%|          | 0/100 [00:00<?, ?it/s]

# 11. Summarising the Regression Statistics

The next step is to summarise the metrics we have just outputted from the k-fold regressions above so we can try and understand which of the regression algorithms has given us the best result.

For this analysis we want to summarise each individual set of outputs for each algorithm and dependent variable and then create summary tables we can use to aid the identification of the algorithm to take forward as the **'best'** and for application to the image data. In this case, we will use the following metrics to summarise the results:

 1. The coefficient of determination (r2),
 2. Root Mean Square Error (RMSE),
 3. Normalised Root Mean Square Error (nRMSE),
 4. Bias.
 5. Normalised Bias

For each algorithm, `rsk.summarise_kfold_metrics` calculates the mean, median, standard deviation and the 5th and 95th percentiles of each metric of each dependent variable over the 100 folds. These are then arranged into a summary table for each dependent variable and metric, with a row per algorithm, which is written to a CSV file:

In [11]:
regress_alg = ["ET", "KR", "EN", "KNN", "PLS", "OLS"]
regress_metrics = [
    et_metrics,
    kr_metrics,
    en_metrics,
    knn_metrics,
    pls_metrics,
    ols_metrics,
]
# The metrics to summarise (the column names in the metrics tables)
sum_metrics = ["r2", "rmse", "nrmse", "bias", "nbias"]

# Summarise the k-fold metrics of each algorithm.
alg_summaries = dict()
for metrics, alg in zip(regress_metrics, regress_alg):
    alg_summaries[alg] = rsk.summarise_kfold_metrics(metrics)
    alg_summaries[alg].to_csv(os.path.join(out_dir, "kfold_{}_summary.csv".format(alg)))

out_summary_stats = dict()
for dep_var in dep_vars:
    dep_var_chk = check_str(
        dep_var, rm_non_ascii=True, rm_dashs=True, rm_spaces=True, rm_punc=True
    ).lower()
    out_summary_stats[dep_var] = dict()
    for stat in sum_metrics:
        # A row for each algorithm with the summary of this metric.
        df_sum_stats = pandas.DataFrame(
            {alg: alg_summaries[alg].loc[(dep_var, stat)] for alg in regress_alg}
        ).T
        df_sum_stats.to_csv(os.path.join(out_dir, "{}_{}_overall_summary.csv".format(dep_var_chk, stat)))
        out_summary_stats[dep_var][stat] = df_sum_stats

# For example, the summary of the normalised RMSE for stem volume:
out_summary_stats["Vol / ha"]["nrmse"].round(3)

,mean,median,std,p05,p95
ET,21.801,21.079,4.193,16.437,29.009
KR,23.142,23.037,4.372,17.383,30.153
EN,22.905,22.755,3.973,17.370,29.634
KNN,26.368,25.711,3.874,20.894,32.693
PLS,24.851,24.787,3.415,19.824,30.246
OLS,23.376,22.767,4.854,17.662,28.963


The previous code created summary files and dataframes, lets now format those into tables for us to interpret, including rounding the numbers to make them more readable. We will also save those tables to CSV files:

In [12]:
summary_median_tabs = dict()

for dep_var in dep_vars:
    sum_stats = dict()
    for stat in out_summary_stats[dep_var]:
        var_df = out_summary_stats[dep_var][stat]
        sum_stats[stat] = var_df["median"].round(3)
    summary_median_tabs[dep_var] = pandas.DataFrame(data=sum_stats)
    dep_var_chk = check_str(
        dep_var, rm_non_ascii=True, rm_dashs=True, rm_spaces=True, rm_punc=True
    ).lower()
    out_csv_file = os.path.join(out_dir, f"{dep_var_chk}_alg_compare_stats.csv")
    summary_median_tabs[dep_var].to_csv(out_csv_file)

Now lets view those summary tables, sorted by the normalised RMSE (try sorting by the other columns):

In [13]:
summary_median_tabs["Mean DBH"].sort_values("nrmse")

,r2,rmse,nrmse,bias,nbias
ET,0.757,3.095,17.813,-0.009,-0.046
KNN,0.736,3.306,18.802,-0.484,-2.849
KR,0.739,3.294,18.847,0.042,0.249
OLS,0.734,3.195,19.037,0.132,0.784
EN,0.718,3.344,19.350,0.095,0.528
PLS,0.690,3.504,20.131,0.084,0.501


In [14]:
summary_median_tabs["BA / ha"].sort_values("nrmse")

,r2,rmse,nrmse,bias,nbias
PLS,0.851,7.440,20.637,0.101,0.310
ET,0.845,7.350,20.853,0.127,0.380
KR,0.840,7.506,21.379,0.016,0.038
OLS,0.831,7.616,21.655,-0.054,-0.152
KNN,0.807,8.136,23.025,-0.800,-2.295
EN,0.796,8.501,24.417,0.054,0.139


In [15]:
summary_median_tabs["Vol / ha"].sort_values("nrmse")

,r2,rmse,nrmse,bias,nbias
ET,0.911,57.930,21.079,-2.548,-0.934
EN,0.906,60.413,22.755,1.176,0.466
OLS,0.901,61.858,22.767,1.771,0.596
KR,0.900,61.206,23.037,1.521,0.614
PLS,0.883,67.992,24.787,1.194,0.468
KNN,0.869,72.337,25.711,-14.067,-5.503


# 12. So, which is 'best'

*Note. The k-fold splits are fixed by `rnd_seed`, so you should get the same results each time; with a different seed the results will differ slightly as the splits will be different.*

Looking at these tables we can see that for Mean DBH the Extra Trees Regressor has produced the best result with a nRMSE of 17.8 % followed by k-Nearest Neighbours with a nRMSE of 18.8 %. For Basal Area, PLSRegression provided the best result (nRMSE: 20.6 %) closely followed by Extra Trees (nRMSE: 20.9 %). While for stand volume Extra Trees also provided the best result (nRMSE: 21.1 %) followed by the ElasticNet and Linear (OLS) regressors (nRMSE: 22.8 %).

Therefore we would take forward the Extra Trees result as the regressor to use for further analysis. However, it is worth noting that

 1. We did not optimise the parameters of the algorithms and if we had done so then the results might have been different - why don't you try to implement this yourself (see notebook 5)?
 2. The LinearRegressor (OLS) is a much simpler model and also produced results which are similar to those of the other algorithms and while it has not produced the best result it is often desirable to use a simpler model.
 3. Be careful of using a model where inputs are outside of the range of values used to train the model, as there is no guarantee that the results will be valid and in some cases the outputs will be completely wrong. However, simpler linear models are less likely to produce values which are completely crazy and therefore might be desirable from that point of view.

Where you have results which are close then it might also be useful to consider the residuals, visualising those to consider the bias (e.g., the k-nearest neighbour regressor has a larger negative bias for stem volume than the others). `plot_residuals` from `rsgispy.regression.regmetrics` can plot the k-fold residuals of several regressors, one row each, for example the stem volume residuals of the Extra Trees and k-nearest neighbour regressors:

In [16]:
# One observed and one predicted column for each regressor.
vol_resid_df = pandas.DataFrame(
    {
        "obs_et": et_residuals["Vol / ha"]["y_true"],
        "pred_et": et_residuals["Vol / ha"]["y_pred"],
        "obs_knn": knn_residuals["Vol / ha"]["y_true"],
        "pred_knn": knn_residuals["Vol / ha"]["y_pred"],
    }
)
fig, axs = plot_residuals(
    vol_resid_df,
    ref_cols=["obs_et", "obs_knn"],
    pred_cols=["pred_et", "pred_knn"],
    target_names=["Vol (m$^3$ ha$^{-1}$), Extra Trees", "Vol (m$^3$ ha$^{-1}$), k-nearest neighbour"],
    title="Stem volume residuals (k-fold)",
)

Both regressors underestimate some of the plots with the largest volumes (the lines of large negative residuals), but the k-nearest neighbour residuals are more often negative for predicted volumes above about 400 m$^3$ ha$^{-1}$, giving its larger negative bias (-15 m$^3$ ha$^{-1}$ over all the folds, against -2.6 m$^3$ ha$^{-1}$ for Extra Trees). A k-nearest neighbour regressor predicts the mean of the nearest training plots, so it cannot predict values beyond those of the plots it averages, and the few plots with very large volumes are averaged with smaller neighbours.